In [24]:
# ---- SETUP -----
# Libraries
import logging
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.dates import DateFormatter
from matplotlib.ticker import MaxNLocator
from scipy import signal
# Logging configuration
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [2]:
# Data retrieving function
def retrieve_data(file_path:str) -> tuple[list[pd.Timestamp], list[float], list[float], list[float], list[float], list[float], list[float], list[float], list[float], list[float], list[float], list[float], list[float]]:
    r"""
    Retrieve data from a text file.

    :param file_path: Path to the text file containing the data.
    :return: A tuple containing lists of data for each column in the file.
    """
    logging.info(f"Retrieving data from {file_path}")
    with open(file_path, 'r') as file:
        data = file.readlines()

    # Process the data and extract relevant columns
    TIME_UTC = []
    BISX = []
    BISY = []
    BISZ = []
    BIST = []
    BOSX = []
    BOSY = []
    BOSZ = []
    BOST = []
    BIS_BOS_X = []
    BIS_BOS_Y = []
    BIS_BOS_Z = []
    BIS_BOS_T = []

    for line in data[1:]:  # Skip header line
        values = line.strip().split(',')
        logging.debug(f"Processing line: {values}")
        TIME_UTC.append(pd.to_datetime(values[0]))
        logging.debug(f"Adding {pd.to_datetime(values[0])}")
        BISX.append(float(values[1]))
        BISY.append(float(values[2]))
        BISZ.append(float(values[3]))
        BIST.append(float(values[4]))
        BOSX.append(float(values[5]))
        BOSY.append(float(values[6]))
        BOSZ.append(float(values[7]))
        BOST.append(float(values[8]))
        BIS_BOS_X.append(float(values[9]))
        BIS_BOS_Y.append(float(values[10]))
        BIS_BOS_Z.append(float(values[11]))
        BIS_BOS_T.append(float(values[12]))

    logging.info("Data retrieval complete.")
    return (TIME_UTC,BISX,BISY,BISZ,BIST,BOSX,BOSY,BOSZ,BOST,BIS_BOS_X,BIS_BOS_Y,BIS_BOS_Z,BIS_BOS_T)

In [5]:
# Data retrieve
file_path = '../data/content/BIO_20060514_DOY134_D001_V1.csv'
TIME_UTC, BISX, BISY, BISZ, BIST, BOSX, BOSY, BOSZ, BOST, BIS_BOS_X, BIS_BOS_Y, BIS_BOS_Z, BIS_BOS_T = retrieve_data(file_path)

2025-06-16 18:28:45,540 - INFO - Retrieving data from ../data/content/BIO_20060514_DOY134_D001_V1.csv
2025-06-16 18:29:26,147 - INFO - Data retrieval complete.


In [ ]:
# Plotting
def plot_component(utc_timestamp: list[pd.Timestamp], component: list[float], title: str, output_file: str) -> None:
    r"""
    Grafica y guarda en `output_filename` el componente `component` contra el tiempo `utc_timestamp`
    
    :param utc_timestamp (list[pd.Timestamp]): Lista de marcas de tiempo.
    :param component (list[float]): Lista de valores del componente a lo largo del tiempo.
    :param output_file (str): Ruta en la que se guarda la gráfica.
    
    :return: None
    """
    STEPS = 1800
    logging.info(f"Plotting component and saving to {output_file}")
    plt.figure(figsize=(100, 30))
    plt.plot(utc_timestamp, component, label='Component', color='red')  #type:ignore
    logging.debug(f"Component data: {component[:5]}...")
    
    # plt.subplots_adjust(left=0.03, right=0.99, top=0.95, bottom=0.15)
    plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True, prune='both', nbins=len(utc_timestamp)//3600 if len(utc_timestamp)//3600 > 0 else 1))
    plt.gca().xaxis.set_major_formatter(DateFormatter('%H:%M'))
    plt.legend(fontsize=32)
    plt.grid(True)
    plt.xticks(np.array(utc_timestamp)[::STEPS], rotation=45, fontsize=32)
    plt.yticks(fontsize=32)
    plt.xlabel('Time (UTC)', fontsize=32)
    plt.ylabel('Magnetic Field (nT)', fontsize=32)
    plt.title(title, fontsize=32)
    
    plt.savefig(output_file)
    plt.close()

In [ ]:
# Plotting originals
plot_component(TIME_UTC, BOSX, "BOSX component", '../data/assets/BOSX.png')
plot_component(TIME_UTC, BOSY, "BOSX component", '../data/assets/BOSY.png')
plot_component(TIME_UTC, BOSZ, "BOSX component", '../data/assets/BOSZ.png')
plot_component(TIME_UTC, BIS_BOS_T, "(BIS-BOS)T figure", '../data/assets/BIS_BOS_T.png')
logging.info("Plots generated successfully.")

2025-06-16 18:49:51,353 - INFO - Plotting component and saving to ../data/assets/BOSX.png


2025-06-16 18:49:52,626 - INFO - Plotting component and saving to ../data/assets/BOSY.png
2025-06-16 18:49:53,850 - INFO - Plotting component and saving to ../data/assets/BOSZ.png
2025-06-16 18:49:55,203 - INFO - Plotting component and saving to ../data/assets/BIS_BOS_T.png
2025-06-16 18:49:56,384 - INFO - Plots generated successfully.


In [32]:
# High-frequency filter
def filtro_pasa_bajas(signal_in: list[float] | np.ndarray, R: float, C: float, sampling_rate: float) -> np.ndarray:
    """
    Modela un filtro pasa bajas RC utilizando scipy.signal.

    :param signal_in (list or np.array): Lista de flotantes representando la señal de entrada.
    :param R (float): Resistencia en Ohms.
    :param C (float): Capacitancia en Faradios.
    :param sampling_rate (float): Frecuencia de muestreo de la señal en Hz.

    Returns:
        list  or np.array: Señal filtrada de salida.
    """
    # Calcula la frecuencia de corte del filtro RC
    f_c = 1 / (2 * np.pi * R * C)
    
    # Normaliza la frecuencia de corte con respecto a la frecuencia de Nyquist
    # (f_nyquist = sampling_rate / 2)
    nyquist_freq = 0.5 * sampling_rate
    normalized_cutoff_freq = f_c / nyquist_freq

    # Diseña el filtro Butterworth de primer orden
    # 'b' son los coeficientes del numerador, 'a' son los coeficientes del denominador
    b, a = signal.butter(1, normalized_cutoff_freq, btype='low', analog=False)  #type: ignore

    # Aplica el filtro a la señal de entrada
    # 'filtfilt' aplica el filtro hacia adelante y hacia atrás para evitar el desfase de fase
    # Si quieres ver el desfase que introduciría el filtro, usa signal.lfilter en su lugar
    filtered_signal = signal.filtfilt(b, a, signal_in)
    
    return filtered_signal
r'''
# --- Ejemplo de uso ---
# Parámetros del circuito RC
R = 720e3  # 720 kOhms
C = 47e-9 # 47 nF

# Parámetros de la señal de entrada
fs = 200e3  # Frecuencia de muestreo: 200 kHz
T_total = 0.02 # Duración total de la señal en segundos
t = np.arange(0, T_total, 1/fs) # Vector de tiempo

# Generar una señal de entrada (ejemplo: onda cuadrada de 5 kHz)
f_signal = 5e3 # Frecuencia de la señal (5 kHz)
amplitude = 1.0 # Amplitud de la señal
square_wave = amplitude * np.sign(np.sin(2 * np.pi * f_signal * t))

# Aplicar el filtro
filtered_signal_scipy = filtro_pasa_bajas(square_wave, R, C, fs)

# --- Visualización ---
plt.figure(figsize=(12, 6))
plt.plot(t * 1000, square_wave, label='Señal de Entrada (Cuadrada)', alpha=0.7)
plt.plot(t * 1000, filtered_signal_scipy, label='Salida del Filtro RC (scipy.signal)', color='green')
plt.title('Simulación de Filtro Pasa Bajas RC (Integrador) con scipy.signal')
plt.xlabel('Tiempo (ms)')
plt.ylabel('Amplitud')
plt.grid(True)
plt.legend()
plt.show()

# Para verificar el comportamiento de integrador:
f_c = 1 / (2 * np.pi * R * C)
print(f"Frecuencia de corte del filtro RC (f_c): {f_c:.2f} Hz")
print(f"Frecuencia de la señal de entrada (f_signal): {f_signal/1000} kHz")
if f_signal > 10 * f_c:
    print("La frecuencia de la señal es significativamente mayor que la frecuencia de corte, el circuito actuará como integrador.")
else:
    print("La frecuencia de la señal no es lo suficientemente alta para un buen comportamiento de integrador.")
'''

'\n# --- Ejemplo de uso ---\n# Parámetros del circuito RC\nR = 720e3  # 720 kOhms\nC = 47e-9 # 47 nF\n\n# Parámetros de la señal de entrada\nfs = 200e3  # Frecuencia de muestreo: 200 kHz\nT_total = 0.02 # Duración total de la señal en segundos\nt = np.arange(0, T_total, 1/fs) # Vector de tiempo\n\n# Generar una señal de entrada (ejemplo: onda cuadrada de 5 kHz)\nf_signal = 5e3 # Frecuencia de la señal (5 kHz)\namplitude = 1.0 # Amplitud de la señal\nsquare_wave = amplitude * np.sign(np.sin(2 * np.pi * f_signal * t))\n\n# Aplicar el filtro\nfiltered_signal_scipy = filtro_pasa_bajas(square_wave, R, C, fs)\n\n# --- Visualización ---\nplt.figure(figsize=(12, 6))\nplt.plot(t * 1000, square_wave, label=\'Señal de Entrada (Cuadrada)\', alpha=0.7)\nplt.plot(t * 1000, filtered_signal_scipy, label=\'Salida del Filtro RC (scipy.signal)\', color=\'green\')\nplt.title(\'Simulación de Filtro Pasa Bajas RC (Integrador) con scipy.signal\')\nplt.xlabel(\'Tiempo (ms)\')\nplt.ylabel(\'Amplitud\')\nplt.gr

In [62]:
# Applying filter
sampling_rate:float = 1.0 # Los datos están registrados cada segundo
f_c: float = 3e-3 # Hz
R = 1e9 # 1T ohm
C = 1 / (2 * np.pi * R * f_c)
logging.info(f"Datos calculados: \n{R=}.\n{C=}.")
hff_BOSX: np.ndarray = filtro_pasa_bajas(BOSX, R, C, 1.0)
hff_BOSY: np.ndarray = filtro_pasa_bajas(BOSY, R, C, 1.0)
hff_BOSZ: np.ndarray = filtro_pasa_bajas(BOSZ, R, C, 1.0)
logging.info(f"signal filtered {hff_BOSX=}")
plot_component(TIME_UTC, list(hff_BOSX), "BOSX filtered", '../data/assets/filtered/BOSX_high_frequency_filter.png')
plot_component(TIME_UTC, list(hff_BOSY), "BOSY filtered", '../data/assets/filtered/BOSY_high_frequency_filter.png')
plot_component(TIME_UTC, list(hff_BOSZ), "BOSZ filtered", '../data/assets/filtered/BOSZ_high_frequency_filter.png')

2025-06-16 20:22:54,307 - INFO - Datos calculados: 
R=1000000000.0.
C=5.305164769729845e-08.
2025-06-16 20:22:54,334 - INFO - signal filtered hff_BOSX=array([13.90709328, 13.98977321, 14.071999  , ..., 19.24194085,
       19.24556558, 19.24864398], shape=(86234,))
2025-06-16 20:22:54,337 - INFO - Plotting component and saving to ../data/assets/filtered/BOSX_high_frequency_filter.png
2025-06-16 20:22:55,351 - INFO - Plotting component and saving to ../data/assets/filtered/BOSY_high_frequency_filter.png
2025-06-16 20:22:57,286 - INFO - Plotting component and saving to ../data/assets/filtered/BOSZ_high_frequency_filter.png
